# 🚀 Gold Mine Trader - Google Colab Setup

**Click Play (▶️) on each cell below to run the system**

This notebook will:
1. Install all libraries
2. Load the trading system
3. Scan for gold mines
4. Show results

**Time needed: 5 minutes**

## STEP 1: Install Libraries

In [ ]:
# Install all required libraries
!pip install --upgrade pip setuptools wheel -q
!pip install yfinance pandas numpy matplotlib requests -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q
!pip install transformers -q

print("✅ All libraries installed!")

## STEP 2: Load the Trading System

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from transformers import pipeline
from datetime import datetime, timedelta
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
import time

# ============================================
# CONFIGURATION
# ============================================

STOCKS = ['AAPL', 'MSFT', 'NVDA', 'TSLA']
GOLD_MINE_THRESHOLD = 0.75
CAPITAL = 500
RISK_PER_TRADE = 0.10

# Technical analysis
MA_SHORT = 20
MA_MEDIUM = 50
MA_LONG = 200
RSI_PERIOD = 14
VOLUME_SPIKE_MULTIPLIER = 1.5

# Scoring weights
WEIGHT_SENTIMENT = 0.35
WEIGHT_CATALYST = 0.30
WEIGHT_TECHNICAL = 0.35

# Catalyst keywords
CATALYST_KEYWORDS = [
    'launch', 'beat', 'earnings', 'partnership', 'deal',
    'upgrade', 'acquisition', 'fda', 'approval', 'profit'
]

print("✅ Configuration loaded!")

## STEP 3: Create Gold Mine Trader Class

In [ ]:
class GoldMineTrader:
    """
    Automated stock trading system that detects "gold mine" opportunities.
    A gold mine = High probability trade when news + technical + catalysts all align.
    """
    
    def __init__(self, symbol='AAPL'):
        self.symbol = symbol
        self.data_cache = {}
        self.cache_time = 0
        self.trades = []
        
        # Load sentiment model (GPU if available)
        device = 0 if torch.cuda.is_available() else -1
        self.sentiment = pipeline(
            'sentiment-analysis',
            model='yiyanghkust/finbert-tone',
            device=device
        )
        
        gpu_status = "✅ GPU" if device == 0 else "⚠️  CPU"
        print(f"🚀 Gold Mine Trader initialized for {symbol}")
        print(f"   Processing: {gpu_status}")
    
    def get_price_data(self, symbol=None):
        """Get cached price data"""
        if symbol is None:
            symbol = self.symbol
        
        if symbol in self.data_cache:
            return self.data_cache[symbol]
        
        data = yf.download(symbol, start='2024-01-01', progress=False)
        self.data_cache[symbol] = data
        return data
    
    def get_news(self, symbol=None):
        """Get latest news about stock"""
        if symbol is None:
            symbol = self.symbol
        
        try:
            url = "https://newsapi.org/v2/everything"
            params = {
                'q': symbol,
                'sortBy': 'publishedAt',
                'language': 'en',
                'pageSize': 3,
                'apiKey': 'demo'
            }
            response = requests.get(url, params=params, timeout=5)
            if response.status_code == 200:
                return response.json().get('articles', [])
        except:
            pass
        return []
    
    def analyze_sentiment(self, news_items):
        """Analyze news sentiment using FinBERT"""
        if not news_items:
            return 0.5
        
        texts = [
            f"{article['title']}. {article['description']}"[:512]
            for article in news_items
        ]
        
        try:
            results = self.sentiment(texts)
            scores = []
            for result in results:
                score = result[0]['score']
                label = result[0]['label']
                if label == 'positive':
                    scores.append(score)
                elif label == 'neutral':
                    scores.append(0.5)
                else:
                    scores.append(1 - score)
            return np.mean(scores)
        except:
            return 0.5
    
    def detect_catalysts(self, news_items):
        """Detect major catalysts (earnings, launches, etc)"""
        if not news_items:
            return 0
        
        count = 0
        for article in news_items:
            text = (article['title'] + ' ' + article['description']).lower()
            if any(keyword in text for keyword in CATALYST_KEYWORDS):
                count += 1
        return count
    
    def calculate_technical_score(self, data):
        """Calculate technical analysis score"""
        if data is None or len(data) < MA_LONG:
            return 0.5
        
        score = 0
        
        # Moving averages
        ma_short = data['Close'].rolling(MA_SHORT).mean().iloc[-1]
        ma_medium = data['Close'].rolling(MA_MEDIUM).mean().iloc[-1]
        ma_long = data['Close'].rolling(MA_LONG).mean().iloc[-1]
        current_price = data['Close'].iloc[-1]
        
        if current_price > ma_short:
            score += 0.15
        if current_price > ma_medium:
            score += 0.15
        if current_price > ma_long:
            score += 0.10
        
        # RSI
        delta = data['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(RSI_PERIOD).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(RSI_PERIOD).mean()
        rs = gain / loss
        rsi = (100 - (100 / (1 + rs))).iloc[-1]
        
        if 30 < rsi < 70:
            score += 0.20
        
        # Volume spike
        vol_avg = data['Volume'].rolling(MA_SHORT).mean().iloc[-1]
        vol_ratio = data['Volume'].iloc[-1] / vol_avg if vol_avg > 0 else 1
        
        if vol_ratio > VOLUME_SPIKE_MULTIPLIER:
            score += 0.25
        
        # Momentum
        if current_price > data['Close'].iloc[-5]:
            score += 0.15
        
        return min(score, 1.0)
    
    def scan_for_gold_mine(self, symbol=None):
        """Complete gold mine detection (2-3 seconds)"""
        if symbol is None:
            symbol = self.symbol
        
        # Parallel data fetching
        with ThreadPoolExecutor(max_workers=2) as executor:
            price_future = executor.submit(self.get_price_data, symbol)
            news_future = executor.submit(self.get_news, symbol)
            
            price_data = price_future.result(timeout=5)
            news_items = news_future.result(timeout=5)
        
        # Parallel analysis
        with ThreadPoolExecutor(max_workers=3) as executor:
            sentiment_future = executor.submit(self.analyze_sentiment, news_items)
            catalysts_future = executor.submit(self.detect_catalysts, news_items)
            technical_future = executor.submit(self.calculate_technical_score, price_data)
            
            sentiment = sentiment_future.result()
            catalysts = catalysts_future.result()
            technical = technical_future.result()
        
        # Calculate score
        catalyst_component = min(catalysts * 0.33, 1.0)
        score = (
            sentiment * WEIGHT_SENTIMENT +
            catalyst_component * WEIGHT_CATALYST +
            technical * WEIGHT_TECHNICAL
        )
        
        return {
            'symbol': symbol,
            'gold_mine_score': score,
            'sentiment': sentiment,
            'catalysts': catalysts,
            'technical': technical,
            'is_gold_mine': score > GOLD_MINE_THRESHOLD,
            'price': price_data['Close'].iloc[-1] if price_data is not None else 0,
            'timestamp': datetime.now()
        }
    
    def scan_multiple_stocks(self, symbols=None):
        """Scan multiple stocks in parallel (8x faster)"""
        if symbols is None:
            symbols = STOCKS
        
        results = {}
        with ThreadPoolExecutor(max_workers=len(symbols)) as executor:
            futures = {
                executor.submit(self.scan_for_gold_mine, symbol): symbol
                for symbol in symbols
            }
            
            for future in as_completed(futures):
                symbol = futures[future]
                try:
                    results[symbol] = future.result()
                except:
                    pass
        
        return results
    
    def print_result(self, result):
        """Print formatted result"""
        score = result['gold_mine_score']
        is_gm = result['is_gold_mine']
        
        print(f"\n{'='*60}")
        print(f"{'🚀 GOLD MINE DETECTED!' if is_gm else '⏳ NOT A GOLD MINE'}")
        print(f"{'='*60}")
        print(f"Stock:           {result['symbol']}")
        print(f"Price:           ${result['price']:.2f}")
        print(f"Gold Mine Score: {score:.2f}/1.00")
        print(f"\nComponents:")
        print(f"  Sentiment:     {result['sentiment']:.2f}")
        print(f"  Catalysts:     {result['catalysts']}")
        print(f"  Technical:     {result['technical']:.2f}")
        print(f"{'='*60}\n")

print("✅ Gold Mine Trader class loaded!")

## STEP 4: Run Your First Scan!

In [ ]:
print("\n" + "="*60)
print("YOUR FIRST GOLD MINE SCAN")
print("="*60 + "\n")

trader = GoldMineTrader(symbol='AAPL')

print("⚡ Scanning AAPL...\n")
start = time.time()

result = trader.scan_for_gold_mine()

elapsed = time.time() - start

print(f"✅ Scan complete in {elapsed:.1f} seconds\n")
trader.print_result(result)

## STEP 5: Scan All 4 Stocks

In [ ]:
print("\n" + "="*60)
print("SCANNING 4 STOCKS IN PARALLEL")
print("="*60 + "\n")

start = time.time()
results = trader.scan_multiple_stocks(STOCKS)
elapsed = time.time() - start

print(f"✅ 4 stocks scanned in {elapsed:.1f} seconds!\n")

gold_mines_found = 0

for symbol, result in results.items():
    trader.print_result(result)
    if result['is_gold_mine']:
        gold_mines_found += 1

print(f"\n🏆 Gold mines found: {gold_mines_found} out of {len(STOCKS)}")

## STEP 6: Continuous Scanning (Optional)

Uncomment below to scan continuously for 5 minutes

In [ ]:
# Uncomment below to run continuous scanning

# print("\n" + "="*60)
# print("CONTINUOUS SCANNING (5 minutes)")
# print("="*60 + "\n")

# scan_count = 0
# gold_mines_found = 0
# start_time = time.time()
# duration = 300  # 5 minutes

# while time.time() - start_time < duration:
#     scan_count += 1
#     current_time = datetime.now().strftime('%H:%M:%S')
#     
#     print(f"[{current_time}] Scan #{scan_count}...")
#     
#     results = trader.scan_multiple_stocks(STOCKS)
#     
#     for symbol, result in results.items():
#         if result['is_gold_mine']:
#             print(f"  🚀 GOLD MINE FOUND: {symbol}!")
#             gold_mines_found += 1
#     
#     time.sleep(30)
# 
# print(f"\nScanning complete!")
# print(f"Total scans: {scan_count}")
# print(f"Gold mines found: {gold_mines_found}")

## Next Steps

✅ **You just ran your first gold mine scan!**

### What to do next:

1. **Run this notebook daily** to monitor for gold mines
2. **Track the results** - Which stocks have high scores?
3. **After 2 weeks** - Open Alpaca account (alpaca.markets)
4. **Paper trade** - Use fake money to validate the system
5. **Go live** - When confident, deposit $500 and start trading

### Understanding the Score:

- **0.75+** = 🚀 GOLD MINE (High confidence buy)
- **0.60-0.75** = 📈 BULLISH (Monitor closely)
- **0.40-0.60** = ⏸️ NEUTRAL (Wait for more signals)
- **<0.40** = 📉 BEARISH (Skip)

### Components Explained:

- **Sentiment** - News analysis (positive/negative)
- **Catalysts** - Major events (earnings, launches, partnerships)
- **Technical** - Price patterns (moving averages, volume, RSI)

---

**Questions?** Check the README: https://github.com/zakdavies2007-hue/gold-mine-trader